**🧹 Làm sạch bộ dữ liệu FULL (rideshare_kaggle.csv)**

Notebook này làm sạch bộ đầy đủ **693.071 chuyến** và xuất ra file sạch để các notebook phân tích dùng lại.

**Đầu vào:** `data/rideshare_kaggle.csv` (350 MB)
**Đầu ra:** `data/rideshare_clean.parquet` (nhẹ & nạp nhanh hơn nhiều)

Các vấn đề sẽ xử lý:
1. Cột `visibility.1` trùng lặp 100% với `visibility`
2. 2 cột cuối bị mất tên header (chỉ có ở bộ rút gọn)
3. `timestamp` đôi khi bị lưu dạng khoa học (`1.55E+09`)
4. **55.095 dòng thiếu giá** (toàn bộ là loại `Taxi`)
5. Chuỗi thừa khoảng trắng
6. Chưa có giờ địa phương & các biến dẫn xuất

> ▶️ **Run All** để chạy toàn bộ. Mất ~30–60 giây.


**1. Nạp dữ liệu thô**

In [ ]:
%matplotlib inline
import warnings, time
import numpy as np, pandas as pd
from pathlib import Path
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)

RAW_NAME   = "rideshare_kaggle.csv"
OUT_NAME   = "dataset_clean.parquet"
_cands = [Path("../data"), Path("data"), Path(".")]
DATA_DIR = next((p for p in _cands if (p/RAW_NAME).exists()), None)
if DATA_DIR is None: raise FileNotFoundError(f"Khong tim thay {RAW_NAME}")
RAW_PATH, OUT_PATH = DATA_DIR/RAW_NAME, DATA_DIR/OUT_NAME
print("Doc tu :", RAW_PATH)
print("Se ghi :", OUT_PATH)

t0 = time.time()
df = pd.read_csv(RAW_PATH)
n_goc, c_goc = df.shape
print(f"\nDa nap trong {time.time()-t0:.1f}s")
print(f"Kich thuoc goc: {n_goc:,} dong x {c_goc} cot")

**2. Chẩn đoán vấn đề (trước khi sửa)**

In [ ]:
print("="*66); print("CHAN DOAN"); print("="*66)

# a) Cot mat ten
unnamed = [c for c in df.columns if str(c).startswith("Unnamed")]
print(f"1) Cot mat ten header      : {unnamed if unnamed else 'khong co'}")

# b) Cot trung lap
if "visibility.1" in df.columns:
    same = (df["visibility"] == df["visibility.1"]).mean()*100
    print(f"2) visibility vs visibility.1: giong nhau {same:.1f}%")

# c) timestamp loi
ts_num = pd.to_numeric(df["timestamp"], errors="coerce")
print(f"3) timestamp khong ep duoc : {ts_num.isna().sum():,} dong")

# d) Thieu gia
n_null_price = df["price"].isna().sum()
print(f"4) Dong thieu gia (price)  : {n_null_price:,} ({n_null_price/len(df)*100:.2f}%)")
if n_null_price:
    print("   Phan bo theo loai dich vu:")
    display(df.loc[df.price.isna(),"name"].value_counts().to_frame("so dong thieu gia"))

# e) Trung lap
print(f"5) Dong trung hoan toan    : {df.duplicated().sum():,}")
print(f"6) id trung                : {df['id'].duplicated().sum():,}")

**3. Bước 1–2: Sửa tên cột & bỏ cột trùng lặp**

In [ ]:
before = df.shape[1]
# 2 cot cuoi bi mat header (chi xuat hien o uber_lyft.csv)
df = df.rename(columns={"Unnamed: 55":"apparentTemperatureMax",
                        "Unnamed: 56":"apparentTemperatureMaxTime"})
# Bo cot trung lap
if "visibility.1" in df.columns:
    df = df.drop(columns=["visibility.1"])
    print("Da bo cot 'visibility.1' (trung 100% voi 'visibility')")
if "timezone" in df.columns and df["timezone"].nunique() <= 1:
    df = df.drop(columns=["timezone"])
    print("Da bo cot 'timezone' (chi co 1 gia tri -> vo dung)")
print(f"So cot: {before} -> {df.shape[1]}")

**4. Bước 3: Sửa `timestamp` & tạo giờ địa phương**

In [ ]:
n_before = len(df)
df["timestamp"] = pd.to_numeric(df["timestamp"], errors="coerce")
df = df.dropna(subset=["timestamp"])
n_bad_ts = n_before - len(df)

df["utc"] = pd.to_datetime(df["timestamp"], unit="s")
n_before = len(df)
df = df[df["utc"] < "2019-01-01"]        # bo moc thoi gian phi ly
n_bad_date = n_before - len(df)

# Boston mua dong = EST = UTC-5 (DST ket thuc 04/11/2018)
df["local"] = df["utc"] - pd.Timedelta(hours=5)

print(f"Bo {n_bad_ts:,} dong timestamp loi")
print(f"Bo {n_bad_date:,} dong moc thoi gian phi ly")
print(f"Khoang thoi gian (local): {df.local.min()}  ->  {df.local.max()}")

**5. Bước 4: Xử lý dòng thiếu giá**
`price` là biến mục tiêu — dòng không có giá thì vô dụng cho mọi phân tích/model, nên **loại bỏ**.

In [ ]:
n_before = len(df)
mat_gia = df[df.price.isna()]
if len(mat_gia):
    print("Cac loai dich vu bi loai bo do khong co gia:")
    display(mat_gia["name"].value_counts().to_frame("so dong"))
df = df.dropna(subset=["price"])
print(f"Bo {n_before-len(df):,} dong thieu gia  ->  con {len(df):,} dong")
print(f"So loai dich vu con lai: {df.name.nunique()}  {sorted(df.name.unique())}")

**Bước 4b: Khử bản ghi trùng lặp**

Có những dòng **giống hệt nhau ở mọi trường** (cùng thời điểm, tuyến, dịch vụ, giá, quãng đường,
thời tiết…), chỉ khác mỗi `id`. Đó là bản ghi bị lưu **2–3 lần** khi thu thập, không phải các
chuyến khác nhau → phải loại bỏ, nếu không sẽ đếm trùng khi thống kê.

> ⚠️ Lưu ý: **không** dùng `timestamp + tuyến + dịch vụ` làm khoá — `timestamp` bị làm tròn nên
> nhiều chuyến **khác nhau thật** (quãng đường khác) lại trùng khoá này.

In [ ]:
n_before = len(df)
cot_khong_id = [c for c in df.columns if c != "id"]
trung = df.duplicated(subset=cot_khong_id, keep=False)

if trung.any():
    print(f"Tim thay {trung.sum():,} dong lien quan den trung lap")
    print("Vi du mot nhom trung:")
    vd = df[trung].copy()
    vd["_k"] = vd.groupby(cot_khong_id, sort=False, dropna=False).ngroup()
    display(vd[vd._k == vd._k.iloc[0]][
        ["id","datetime","source","destination","name","price","distance","temperature"]])

df = df.drop_duplicates(subset=cot_khong_id, keep="first")
print(f"\nDa bo {n_before-len(df):,} ban sao thua  ->  con {len(df):,} dong")

# Kiem tra lai
assert not df.duplicated(subset=cot_khong_id).any(), "Van con trung lap!"
assert not df["id"].duplicated().any(), "Con id trung!"
print("[OK] Khong con ban ghi trung lap.")

**6. Bước 5: Chuẩn hoá chuỗi**

In [ ]:
for c in ["short_summary","long_summary","icon","source","destination","name","cab_type"]:
    if c in df.columns:
        df[c] = df[c].astype(str).str.strip()
print("Da bo khoang trang thua o cac cot chuoi.")
print("Vi du short_summary:", sorted(df.short_summary.unique())[:6])

**7. Bước 6: Tạo biến dẫn xuất**
Các cột này **không có trong dữ liệu gốc** — được tính thêm để phục vụ phân tích.

In [ ]:
df["hour_local"]     = df["local"].dt.hour
df["weekday_local"]  = df["local"].dt.weekday          # 0=T2 ... 6=CN
df["date_local"]     = df["local"].dt.normalize()
df["is_weekend"]     = (df["weekday_local"] >= 5).astype(int)
df["price_per_mile"] = np.where(df["distance"] > 0, df["price"]/df["distance"], np.nan)
df["is_surge"]       = (df["surge_multiplier"] > 1).astype(int)

moi = ["utc","local","hour_local","weekday_local","date_local",
       "is_weekend","price_per_mile","is_surge"]
print(f"Da tao {len(moi)} cot moi:")
for c in moi: print("   -", c)
df[moi].head(3)

**8. Bước 7: Kiểm tra chất lượng sau làm sạch**
Xác nhận dữ liệu đã sạch trước khi lưu.

In [ ]:
loi = []
if df.price.isna().any():                 loi.append("Van con dong thieu gia")
if (df.price <= 0).any():                 loi.append("Co gia <= 0")
if (df.distance < 0).any():               loi.append("Co quang duong am")
if df.duplicated().any():                 loi.append("Con dong trung lap")
if df.duplicated(subset=[c for c in df.columns if c!="id"]).any():
                                          loi.append("Con ban ghi trung (khac moi id)")
if df["id"].duplicated().any():           loi.append("Con id trung")
if (df.surge_multiplier < 1).any():       loi.append("Co surge < 1")
if df.local.isna().any():                 loi.append("Con thoi gian NaT")

print("="*66); print("KIEM TRA CHAT LUONG"); print("="*66)
if loi:
    for e in loi: print("  [X]", e)
else:
    print("  [OK] Tat ca kiem tra deu dat.")

print()
print(f"Gia      : {df.price.min():.2f} -> {df.price.max():.2f}  (TB {df.price.mean():.2f})")
print(f"Quang duong: {df.distance.min():.2f} -> {df.distance.max():.2f}")
print(f"Surge    : {df.surge_multiplier.min()} -> {df.surge_multiplier.max()}")
print(f"Con thieu du lieu o {int((df.isna().sum()>0).sum())} cot")

In [ ]:
# CANH BAO QUAN TRONG: surge theo hang
t = df.groupby("cab_type")["surge_multiplier"].agg(
        so_cuoc="size", lon_nhat="max", so_surge=lambda s:(s>1).sum())
t["ty_le_%"] = (t.so_surge/t.so_cuoc*100).round(2)
display(t)
for h in df.cab_type.unique():
    if df.loc[df.cab_type==h,"surge_multiplier"].max() == 1.0:
        print(f"!! CANH BAO: {h} co surge = 1.0 o TAT CA {(df.cab_type==h).sum():,} dong.")
        print(f"   -> Day la HAN CHE CUA DU LIEU (API {h} khong tra ve truong nay).")
        print(f"   -> Moi phan tich ve surge PHAI loc: df[df.cab_type != '{h}']")

**9. Lưu file sạch**

In [ ]:
t0 = time.time()
df.to_parquet(OUT_PATH, index=False)
sz_in  = RAW_PATH.stat().st_size/1048576
sz_out = OUT_PATH.stat().st_size/1048576
print(f"Da luu: {OUT_PATH}  ({time.time()-t0:.1f}s)")
print(f"Dung luong: {sz_in:.0f} MB (CSV goc)  ->  {sz_out:.0f} MB (parquet sach)")
print(f"Giam {100*(1-sz_out/sz_in):.0f}%")

# Kiem tra doc lai duoc
t0=time.time(); chk = pd.read_parquet(OUT_PATH)
print(f"\nDoc lai kiem tra: {len(chk):,} dong x {chk.shape[1]} cot trong {time.time()-t0:.1f}s")
assert len(chk)==len(df) and chk.shape[1]==df.shape[1], "File luu bi sai!"
print("[OK] File sach hop le.")

**10. Tổng kết trước / sau**

In [ ]:
print("="*66); print("TONG KET LAM SACH"); print("="*66)
print(f"{'':22}{'TRUOC':>14}{'SAU':>14}")
print(f"{'So dong':22}{n_goc:>14,}{len(df):>14,}")
print(f"{'So cot':22}{c_goc:>14}{df.shape[1]:>14}")
print()
print("Da xu ly:")
print(f"  - Bo cot trung lap visibility.1")
print(f"  - Bo {n_goc-len(df):,} dong ({(n_goc-len(df))/n_goc*100:.1f}%) khong co gia (loai Taxi)")
print(f"  - Chuan hoa chuoi, tao gio dia phuong (UTC-5)")
print(f"  - Them 8 cot dan xuat")
print()
print(f"Ket qua: {len(df):,} chuyen sach, {df.date_local.nunique()} ngay, "
      f"{df.name.nunique()} loai dich vu, {df.source.nunique()} khu vuc")
print(f"Cuoc co surge (chi Lyft): {df.is_surge.sum():,}")
print()
print(f">> Dung file nay o cac notebook khac:")
print(f"   df = pd.read_parquet('../data/{OUT_NAME}')")